In [1]:
"""
المرحلة 5.1 + 5.2:
- تجميع الملامح على مستوى الفيديو (Feature Vector) كما في Algorithm 3.1
- بناء PoC Dataset للتدريب
"""
import pandas as pd
import numpy as np
import os

# ==========================================
# 5.1: تجميع الملامح على مستوى الفيديو (Feature Vector)
# يطبق Algorithm 3.1 من التقرير
# ==========================================
def aggregate_video_features(csv_path, subject_age=7):
    """
    يحول بيانات الإطار-بإطار إلى Feature Vector واحد للفيديو.
    FeatureVector = [Mean(KneeAngle_L), Mean(KneeAngle_R), Cadence, Symmetry_Knee, Age]
    """
    df = pd.read_csv(csv_path)
    
    # حساب المتوسطات (حسب Algorithm 3.1)
    mean_knee_L = df["L_Knee_Filtered"].mean()
    mean_knee_R = df["R_Knee_Filtered"].mean()
    mean_symmetry = df["Knee_Symmetry_Filtered"].mean()
    
    # تقدير Cadence: عدد دورات المشي في الدقيقة
    # نستخدم تقلبات زاوية الركبة اليسرى لاكتشاف دورات المشي
    knee_signal = df["L_Knee_Filtered"].dropna().values
    if len(knee_signal) > 30:
        # عتبة بسيطة: القيمة أقل من المتوسط تعتبر "مرحلة وقوف"
        threshold = np.mean(knee_signal) - 0.3 * np.std(knee_signal)
        # عد التحولات من فوق العتبة إلى تحتها (تقريب لعدد الخطوات)
        crossings = np.sum(np.diff(knee_signal < threshold).astype(int) == 1)
        duration_sec = df["Timestamp_sec"].iloc[-1] - df["Timestamp_sec"].iloc[0]
        cadence = (crossings / duration_sec * 60) if duration_sec > 0 else 0
    else:
        cadence = 0
    
    return {
        "Mean_KneeAngle_L": mean_knee_L,
        "Mean_KneeAngle_R": mean_knee_R,
        "Cadence": cadence,
        "Symmetry_Knee": mean_symmetry,
        "Age": subject_age,
        "Source_File": os.path.basename(csv_path)
    }


# ==========================================
# 5.2: بناء PoC Dataset (Synthetic Data for Concept Validation)
# ==========================================
def generate_poc_dataset(n_typical=40, n_atypical=40, seed=42):
    """
    يولّد مجموعة بيانات تجريبية (PoC) بناءً على الأدبيات:
    - Typical: تماثل < 10%, Cadence 90-110, زوايا متقاربة
    - Atypical: تماثل > 15%, Cadence غير منتظم, زوايا مختلفة
    
    ملاحظة: هذه بيانات PoC للتحقق من المفهوم فقط.
    يجب استبدالها بـ dataset حقيقي (مثل Ganai et al.) قبل التقييم النهائي.
    """
    np.random.seed(seed)
    data = []
    
    # --- Typical Gait (بناءً على Kidziński et al., 2020) ---
    for i in range(n_typical):
        knee_L = np.random.normal(145, 8)     # زوايا متقاربة
        knee_R = np.random.normal(145, 8)
        cadence = np.random.normal(100, 8)    # 90-110 steps/min
        symmetry = np.random.uniform(2, 9)    # < 10%
        age = np.random.randint(4, 11)
        data.append({
            "Mean_KneeAngle_L": knee_L,
            "Mean_KneeAngle_R": knee_R,
            "Cadence": cadence,
            "Symmetry_Knee": symmetry,
            "Age": age,
            "Label": "Typical"
        })
    
    # --- Atypical Gait (بناءً على Ganai et al., 2025 - ASD patterns) ---
    for i in range(n_atypical):
        knee_L = np.random.normal(135, 15)    # تباين أكبر
        knee_R = np.random.normal(155, 15)    # فرق واضح بين الجانبين
        cadence = np.random.normal(85, 15)    # أبطأ وأكثر تبايناً
        symmetry = np.random.uniform(12, 25)  # > 10% (Robinson threshold)
        age = np.random.randint(4, 11)
        data.append({
            "Mean_KneeAngle_L": knee_L,
            "Mean_KneeAngle_R": knee_R,
            "Cadence": cadence,
            "Symmetry_Knee": symmetry,
            "Age": age,
            "Label": "Atypical"
        })
    
    return pd.DataFrame(data)


# ==========================================
# التشغيل الرئيسي
# ==========================================
if __name__ == "__main__":
    # 1) تجميع الفيديو الحقيقي (إن وُجد)
    if os.path.exists("gait_features_clean.csv"):
        video_features = aggregate_video_features("gait_features_clean.csv", subject_age=7)
        print("📊 Feature Vector للفيديو الحقيقي:")
        print(pd.DataFrame([video_features]).T)
        pd.DataFrame([video_features]).to_csv("real_video_features.csv", index=False)
    
    # 2) بناء PoC Dataset
    poc_df = generate_poc_dataset(n_typical=40, n_atypical=40)
    poc_df.to_csv("poc_training_dataset.csv", index=False)
    
    print(f"\n✅ تم إنشاء PoC Dataset: {len(poc_df)} عينة")
    print(poc_df["Label"].value_counts())
    print("\n📋 معاينة:")
    print(poc_df.head())

📊 Feature Vector للفيديو الحقيقي:
                                        0
Mean_KneeAngle_L                134.36011
Mean_KneeAngle_R               123.986665
Cadence                         45.833333
Symmetry_Knee                    20.30437
Age                                     7
Source_File       gait_features_clean.csv

✅ تم إنشاء PoC Dataset: 80 عينة
Label
Typical     40
Atypical    40
Name: count, dtype: int64

📋 معاينة:
   Mean_KneeAngle_L  Mean_KneeAngle_R     Cadence  Symmetry_Knee  Age    Label
0        148.973713        143.893886  105.181508       3.092130    6  Typical
1        157.184239        144.243032   92.569375       4.335960    6  Typical
2        141.244205        149.340480   96.292658       4.129696    9  Typical
3        141.274162        125.487153  104.827530       2.326660    7  Typical
4        139.373250        127.883035   94.964200       5.599641    4  Typical
